## 一、工具

### 工具 1: 安全的文件读取 (The Eyes)

In [1]:
from langchain_core.tools import tool
from langchain_community.tools.file_management import ReadFileTool

C:\Users\yanchen\AppData\Local\Temp\ipykernel_38916\1363220637.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.file_management import ReadFileTool


In [2]:
# 初始化底层工具，强制指定 root_dir 为项目根目录，只能读取 skills下的文件
raw_read_file = ReadFileTool(root_dir="./skills")

In [3]:
# 这个tool包装官方tool实现

@tool
def read_file(file_path: str) -> str:
    """
    用于读取 Agent Skills 的定义文件 (SKILL.md)。
    输入参数 file_path 必须是相对于 skills 文件夹的路径，例如: 'get_weather/SKILL.md'
    """
    try:    
        return raw_read_file.invoke({"file_path": file_path})
    except Exception as e:
        return f"Error reading file: {str(e)}"

### 工具 2: 联网能力

In [4]:
from langchain_tavily import TavilySearch
import os 
os.environ["TAVILY_API_KEY"] =''

search = TavilySearch(max_results=2)
result = search.invoke("今天上海的天气怎么样")
result

{'query': '今天上海的天气怎么样',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://chinaweather.org/zh-hans/shanghai',
   'title': '上海 今日天气预报',
   'content': '# 上海， 中国 天气. ## 目前情况. 风: **WNW - 2 km/h**. ## 天气预报. ## 30 天平均值和极值. ## 昨天的数据. ## 未来 12 小时内上海的气温和降雨可能性. ## 未来几天上海的气温和降雨可能性. ## 未来几天上海将有降雨. ## 上海 Climate Summary. 上海 features a None (Köppen classification: Cfa), with an average annual temperature of 18.33ºC (64.99ºF), which is about 高于 3.71% the national average for 中国. 该市每年都会经历约 67.1 毫米（2.64 英寸）的降雨量，分布在 111.71 个雨天，占全年的 30.61%。. | 常年高温 | 20.46ºC (68.83ºF) |. | 最热月份 | 六月 27.13ºC (80.83ºF) |. | 年降水量 | 67.1mm (2.64in) |. | 有降雨的日子 | 111.71 天 (30.61%) |. ## 每月天气平均值. | 月 | 高/低（°C） | 雨 |. | 行进 | 14.17° / 6.97° | 9.64 天 |. ## 上海的天气. ## 常见问题. 六月 的月平均气温最高（日平均值为 27.13°C），一月 的月平均气温最低（日平均值为 3.48°C）。. 今天 上海 天气非常炎热，气温高于 25°C (77.0°F)，请考虑穿：. ### 温度. ### 日出/日落. ### 上海 气象图. 版权 © 2026 Chinaweather.Org - 天气和气候网络成员.',
   'score': 0.8380581,
   'raw_content': None},
  {'u

In [5]:
# 将工具打包，准备注入 Agent
core_tools = [read_file, search]

## 二、技能引导层 (Skill Bootstrap) 实现

Agent 怎么知道自己有哪些技能？我们需要在系统启动时，通过代码自动扫描目录。这被称为Bootstrap (引导过程)。我们将编写一个 bootstrap.py，它的作用是生成一份技能菜单：

In [6]:
import os

def generate_skills_snapshot(skills_dir="./skills"):
    """
    扫描 skills 文件夹，提取元数据，生成 XML 格式的快照
    """
    
    # 使用 XML 标签包裹，有助于 Claude/GPT 模型更精准地识别结构
    snapshot = "<available_skills>\n"
    
    # 1. 遍历 skills 目录下的所有子文件夹
    # 假设每个子文件夹就是一个 Skill
    if not os.path.exists(skills_dir):
        return "<available_skills>No skills found.</available_skills>"

    for skill_name in os.listdir(skills_dir):
        skill_path = os.path.join(skills_dir, skill_name, "SKILL.md")
        
        # 2. 校验：只有包含 SKILL.md 的文件夹才被视为有效技能
        if os.path.exists(skill_path):
            
            # 3. 读取 SKILL.md 的内容（实际项目中建议只读前几行元数据）
            with open(skill_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # 找到描述
            description = None
            for line in content.split('\n'):
                line = line.strip()
                if line.startswith('description:'):
                    description = line.replace('description:', '').strip()
                    break  

            
            # 简单提取：假设 SKILL.md 第一行是 name，第二行是 description
            # 实际生产中应使用 YAML Frontmatter 解析库
            snapshot += f"""  <skill>
    <name>{skill_name}</name>
    <path>{skill_name}/SKILL.md</path>
    <description>{description}</description>
  </skill>\n"""
            
    snapshot += "</available_skills>"
    return snapshot

In [7]:
x = generate_skills_snapshot()
print(x)

<available_skills>
  <skill>
    <name>get_weather</name>
    <path>get_weather/SKILL.md</path>
    <description>获取指定城市的实时天气信息</description>
  </skill>
  <skill>
    <name>summary</name>
    <path>summary/SKILL.md</path>
    <description>对一段文章或网页进行总结</description>
  </skill>
</available_skills>


## 三、Agent 运行时构建 (LangChain 1.x Standard)

在 LangChain 1.x 中，官方引入了基于 Graph 理念的 create_agent API，它自动封装了“思考-工具调用-再思考”的复杂循环。

In [8]:
from langchain.agents import create_agent
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import SystemMessage

In [9]:
# 1. 初始化模型
model = ChatDeepSeek(model="deepseek-chat",
                     api_key = '',
                     base_url = 'https://api.deepseek.com')

# query = "你好，请你介绍一下你自己。"
# response = model.invoke(query)
# print(response.content)

In [10]:
# 2. 动态构建 System Prompt (核心魔法)
# 将刚才生成的 Skill Snapshot 注入到提示词中
snapshot_content = generate_skills_snapshot()


# 这段 Prompt 决定了 Agent 的行为模式：是直接回答，还是去查文档
system_instruction = f"""你是一个基于 Agent Skills 架构的智能助手。

【能力清单】
你拥有以下技能列表 (SKILLS_SNAPSHOT)，它们定义了你的专业能力：
{snapshot_content}

【核心协议 (CRITICAL)】
1. 当用户请求的任务匹配上述技能时，你严禁直接猜测操作步骤。
2. 你的第一步行动必须是调用 `read_file` 工具，读取该技能对应的 `<path>` 文件。
3. 读取文件后，严格按照文件中的 Steps (步骤) 指导去执行任务。
"""

print(system_instruction)

你是一个基于 Agent Skills 架构的智能助手。

【能力清单】
你拥有以下技能列表 (SKILLS_SNAPSHOT)，它们定义了你的专业能力：
<available_skills>
  <skill>
    <name>get_weather</name>
    <path>get_weather/SKILL.md</path>
    <description>获取指定城市的实时天气信息</description>
  </skill>
  <skill>
    <name>summary</name>
    <path>summary/SKILL.md</path>
    <description>对一段文章或网页进行总结</description>
  </skill>
</available_skills>

【核心协议 (CRITICAL)】
1. 当用户请求的任务匹配上述技能时，你严禁直接猜测操作步骤。
2. 你的第一步行动必须是调用 `read_file` 工具，读取该技能对应的 `<path>` 文件。
3. 读取文件后，严格按照文件中的 Steps (步骤) 指导去执行任务。



In [11]:
# 3. 构建 Agent Runtime
# create_agent 自动将 model 和 tools 绑定，并处理工具调用的循环逻辑
agent_executor = create_agent(
    model=model,
    tools=core_tools,
    
    # 直接传入动态构建的 Prompt
    system_prompt=system_instruction,
)

In [16]:
# 4. 执行测试：模拟用户提问
query = "帮我查一下北京现在的天气"
# query = "帮我总结一下什么是openclaw"



if __name__ == "__main__":
    
    print(">>> Agent System Initialized.")
    print(">>> User: 帮我查一下北京现在的天气")
    
    # stream 方法允许我们看到 Agent 的思考过程
    # inputs 是用户输入，config 可以传入会话 ID
    events = agent_executor.stream(
        {"messages": [("user", query)]},
        stream_mode="values"
    )
    
    for event in events:
        # 打印交互过程，方便调试
        if "messages" in event:
            last_msg = event["messages"][-1]
            print(f"[{last_msg.type.upper()}]: {last_msg.content}")

>>> Agent System Initialized.
>>> User: 帮我查一下北京现在的天气
[HUMAN]: 帮我查一下北京现在的天气
[AI]: 好的，我先读取天气技能的定义文件，了解具体的操作步骤。
[TOOL]: name: get_weather
description: 获取指定城市的实时天气信息


# 获取天气技能 (get_weather)


## 使用场景
当用户询问某个城市的天气情况时，使用此技能。

## 执行步骤

1. **提取城市名称**：从用户消息中准确提取目标城市名称并生成问答，（例如：“北京今天天气怎么样？”或“查询上海的天气”）
2. **调用工具获取数据**：调用search 工具进行查询
3. **解析 JSON 数据**：从返回的数据包中提取以下核心指标：
    * **当前温度**
    * **天气状况**
    * **湿度**
    * **风速**
4. **生成回复**：用简洁的自然语言向用户汇总并汇报天气情况。


## 示例

**用户：** 「查询北京的天气」

**执行流程：**
1. **提取城市：** 北京 (Beijing)
2. **调用：** ` search("北京今天天气怎么样")`
3. **解析 JSON 结果**
4. **回复格式：** xx当前天气：晴，温度 25°C，湿度 40%，东南风 3 级。
[AI]: 好的，根据技能文件的说明，我来查询北京的实时天气。
[TOOL]: {"query": "北京今天实时天气 温度 湿度 风速", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.aqi.in/weather/cn/china/beijing/beijing/beijing-us-embassy", "title": "实时Beijing US Embassy天气状况：气温| 30天预报 - AQI.in", "content": "View aqi PageAQIView weather Page天气. # Beijing US Embassy Weather Conditions Current Temperatu

In [17]:
from pprint import pprint

for event in agent_executor.stream(
    {"messages": [("user", query)]},
    stream_mode="updates"
):
    pprint(event)
    print("=" * 100)


{'model': {'messages': [AIMessage(content='好的，我来查看相关技能的定义文件。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 2100, 'total_tokens': 2159, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2048}, 'prompt_cache_hit_tokens': 2048, 'prompt_cache_miss_tokens': 52}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'b943e9fd-1820-4bf5-bf08-9fe147478f4f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1c3a-9de1-79b1-8893-131840ba568d-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': 'get_weather/SKILL.md'}, 'id': 'call_00_ud9UTodkoE7xAqMagjYo4199', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2100, 'output_tokens': 59, 'total_tokens': 2159, 'input_token_details': {'cache_read': 2048}, 'output_token_details': {}})]}}
{'tools': {'

In [ ]:
# 4. 执行测试：模拟用户提问
# query = "帮我查一下北京现在的天气"
query = "帮我总结一下什么是openclaw"



if __name__ == "__main__":

    print(">>> Agent System Initialized.")
    print(">>> User: 帮我总结一下什么是openclaw")

    # stream 方法允许我们看到 Agent 的思考过程
    # inputs 是用户输入，config 可以传入会话 ID
    events = agent_executor.stream(
        {"messages": [("user", query)]},
        stream_mode="values"
    )

    for event in events:
        # 打印交互过程，方便调试
        if "messages" in event:
            last_msg = event["messages"][-1]
            print(f"[{last_msg.type.upper()}]: {last_msg.content}")

# 笔记
## tool与skill
Skill 不是直接执行能力，而是 Agent 在执行任务前读取的“操作说明书”。真正执行的仍然是 Tool；状态流转仍然是 HumanMessage → AIMessage(tool_call) → ToolMessage → AIMessage，只是 Skill Agent 会先多一次 read_file(SKILL.md)

## 传统agent与skills agent
传统 Agent：
模型直接决定怎么做，
模型每次都临时规划，容易不稳定。

Skills Agent：
模型先查技能说明，再按照说明做
把“任务怎么做”写到：SKILL.md里。模型每次先读取技能文件，再执行。这就把一部分 prompt 固化成了外部文档：
1技能说明文档化，2执行流程可维护，3行为更稳定